# 🔬 Notebook 3: Airbnb — Deep Dive

## 🛠️ Setup

```bash
cd 06-system-designs/airbnb
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

In this notebook we walk four real problems with a **bad → better → best** progression and runnable code:

1. Preventing double-bookings under concurrency
2. Fast geo search
3. Caching hot listings
4. Rate-limiting search

## 1️⃣ Preventing double-bookings

The enemy is the interleaving:

```
T1: SELECT  availability → free
T2: SELECT  availability → free
T1: INSERT  booking      ✅
T2: INSERT  booking      ✅   ← DOUBLE BOOK
```

### 🐌 Bad — no coordination

In [1]:
import threading, time
from datetime import date, timedelta

# Shared state: booked days per listing. No lock.
booked: dict[int, set] = {}

def book_no_lock(listing_id, days):
    # simulate the read-then-write race
    current = booked.setdefault(listing_id, set())
    if current & set(days):
        return False
    time.sleep(0.001)        # <-- widens the race window
    current.update(days)
    return True

days = [date(2026,5,1) + timedelta(days=i) for i in range(3)]
results = []

def worker(i):
    results.append(book_no_lock(1, days))

ts = [threading.Thread(target=worker, args=(i,)) for i in range(10)]
for t in ts: t.start()
for t in ts: t.join()

succ = sum(results)
print(f"successes: {succ}  (we expected exactly 1)")

successes: 10  (we expected exactly 1)


Running 10 threads, several "succeed" — each thinks the calendar is empty. That's a double-booking.

### ✅ Better — in-process lock

A mutex serialises the check-and-update. Correct in a single process, but useless across multiple servers.

In [2]:
class Calendar:
    def __init__(self):
        self._booked: set[tuple[int, date]] = set()
        self._lock = threading.Lock()

    def book(self, listing_id, days):
        with self._lock:
            keys = {(listing_id, d) for d in days}
            if self._booked & keys:
                return False
            self._booked |= keys
            return True

cal = Calendar()
results = []
def w(i):
    results.append(cal.book(1, days))

ts = [threading.Thread(target=w, args=(i,)) for i in range(10)]
for t in ts: t.start()
for t in ts: t.join()
print(f"successes: {sum(results)}  (exactly 1 ✔)")

successes: 1  (exactly 1 ✔)


### 🏆 Best — let the database do it

In production you have many app servers, and a mutex in Python RAM means nothing to another process. Push the invariant down to the database with a **unique constraint**. Below: same logic, backed by SQLite, concurrent threads both issuing transactions.

In [3]:
import sqlite3, threading
from datetime import date, timedelta

con = sqlite3.connect(":memory:", check_same_thread=False, isolation_level=None)
con.execute("PRAGMA journal_mode=WAL")
con.execute('''
    CREATE TABLE availability(
        listing_id INTEGER, day TEXT, booking_id INTEGER,
        PRIMARY KEY (listing_id, day)
    )
''')

lock = threading.Lock()  # SQLite allows only one writer at a time
results: list[bool] = []

def try_book(guest_id: int):
    days_iso = [(date(2026,5,1)+timedelta(days=i)).isoformat() for i in range(3)]
    with lock:
        try:
            con.execute("BEGIN IMMEDIATE")
            for d in days_iso:
                con.execute(
                    "INSERT INTO availability(listing_id,day,booking_id) VALUES (1,?,?)",
                    (d, guest_id),
                )
            con.execute("COMMIT")
            results.append(True)
        except sqlite3.IntegrityError:
            con.execute("ROLLBACK")
            results.append(False)

ts = [threading.Thread(target=try_book, args=(i,)) for i in range(10)]
for t in ts: t.start()
for t in ts: t.join()

print(f"successes: {sum(results)}  (exactly 1 ✔)")
print("winner had guest_id =",
      con.execute("SELECT DISTINCT booking_id FROM availability").fetchone()[0])

successes: 1  (exactly 1 ✔)
winner had guest_id = 0


> In Postgres you'd use `INSERT ... ON CONFLICT DO NOTHING RETURNING` or wrap the check-insert in a `SERIALIZABLE` transaction. The idea is identical: **the database is the arbiter**.

### Trade-offs

| Option | Pros | Cons |
|---|---|---|
| App mutex | easy | single-process only; useless at scale |
| Distributed lock (Redis/ZK) | works across nodes | extra infra; tricky edge cases |
| DB unique constraint | simplest; correct | DB is the bottleneck (fine here) |
| Optimistic concurrency | great for rare conflicts | retries needed |

## 2️⃣ Geo search

Users search by location. Brute-force linear scan of 10M listings per request is dead on arrival.

### 🐌 Bad — scan every listing

In [4]:
import math, random, time

random.seed(0)
# 200k synthetic listings scattered around the world
N = 200_000
listings = [(i, random.uniform(-90, 90), random.uniform(-180, 180)) for i in range(N)]

def haversine(lat1, lng1, lat2, lng2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2*R*math.asin(math.sqrt(a))

def scan_nearby(lat, lng, radius_km=50):
    return [lid for lid, la, lo in listings if haversine(lat, lng, la, lo) <= radius_km]

t0 = time.time()
hits = scan_nearby(47.6, -122.3)
print(f"linear scan: {len(hits)} hits in {(time.time()-t0)*1000:.0f} ms over {N:,} listings")

linear scan: 1 hits in 44 ms over 200,000 listings


### ✅ Better — bucket by coarse grid (poor-man's geohash)

Round (lat, lng) to a small number of decimals → a grid cell. Index listings by cell; at query time scan only the target cell plus its 8 neighbours.

In [5]:
from collections import defaultdict

# 1 degree of latitude ≈ 111 km, so for a 50 km radius a 1-degree grid
# with a 3×3 neighbour scan comfortably covers the query circle.
def cell(lat, lng, precision=0):
    step = 10 ** -precision
    return (round(lat/step)*step, round(lng/step)*step)

index = defaultdict(list)
for lid, la, lo in listings:
    index[cell(la, lo)].append((lid, la, lo))

def grid_nearby(lat, lng, radius_km=50, precision=0):
    step = 10 ** -precision
    candidates = []
    for dlat in (-step, 0, step):
        for dlng in (-step, 0, step):
            candidates += index.get(cell(lat+dlat, lng+dlng, precision), [])
    return [lid for lid, la, lo in candidates
            if haversine(lat, lng, la, lo) <= radius_km]

t0 = time.time()
hits = grid_nearby(47.6, -122.3)
print(f"grid index : {len(hits)} hits in {(time.time()-t0)*1000:.0f} ms")

grid index : 1 hits in 0 ms


### 🏆 Best — hierarchical cells (S2 / H3) in a real search engine

Production systems use **S2** (Google) or **H3** (Uber) cells, or geohash prefixes, stored in Elasticsearch/OpenSearch with `geo_point` fields. Two big wins:

- **Multi-resolution** — coarse cells for zoomed-out maps, fine cells for "within this block".
- **Pre-filtering** — the search engine can combine geo + dates + amenities in one query.

The grid demo above captures the *idea*; a library like `h3-py` or the ES `geo_distance` query gives you the polished version.

> Tip: always filter by **date availability first** (cheap index lookup), then by geo (more work), then rank. Cheapest filters first.

## 3️⃣ Caching hot listings

100× more reads than writes → cache aggressively. The popular cache patterns:

- **Cache-aside** (lazy): app reads cache; miss → read DB → fill cache.
- **Write-through**: writes go to cache and DB.
- **TTL**: entries expire so stale data eventually corrects.

Two cache traps:

- **Stampede** — TTL expires, 1000 concurrent requests all miss and hit the DB. Fix with **single-flight** (only one request fetches, the others wait) or **stale-while-revalidate**.
- **Inconsistency** — booking just mutated availability, but cached listing detail still shows those dates free. Fix: invalidate on write, or cache only with short TTL.

In [6]:
import time, threading

class TTLCache:
    def __init__(self, ttl=2.0):
        self.ttl = ttl
        self._d: dict = {}
        self._lock = threading.Lock()
        self._inflight: dict = {}

    def get(self, key, loader):
        now = time.time()
        with self._lock:
            hit = self._d.get(key)
            if hit and hit[1] > now:
                return hit[0], "HIT"
            # single-flight: only one loader per key
            ev = self._inflight.get(key)
            if ev is None:
                ev = threading.Event()
                self._inflight[key] = ev
                leader = True
            else:
                leader = False

        if leader:
            try:
                val = loader()
                with self._lock:
                    self._d[key] = (val, now + self.ttl)
                return val, "MISS-LEADER"
            finally:
                with self._lock:
                    self._inflight.pop(key, None)
                ev.set()
        else:
            ev.wait(timeout=5)
            return self._d[key][0], "MISS-FOLLOWER"

db_calls = 0
def load_listing():
    global db_calls
    db_calls += 1
    time.sleep(0.05)        # simulate DB latency
    return {"id": 1, "title": "Cozy cabin", "price_cents": 12000}

cache = TTLCache(ttl=1.0)
results: list = []

def reader(i):
    val, status = cache.get("listing:1", load_listing)
    results.append(status)

ts = [threading.Thread(target=reader, args=(i,)) for i in range(20)]
for t in ts: t.start()
for t in ts: t.join()

from collections import Counter
print(Counter(results))
print(f"DB calls: {db_calls}   (target: 1)")

Counter({'MISS-FOLLOWER': 19, 'MISS-LEADER': 1})
DB calls: 1   (target: 1)


With single-flight the **DB is hit exactly once** even when 20 threads all miss at the same time.

Real-world deployment:

- Put **Redis** in front of the listing service. TTL 60–300s for listing detail.
- Put a **CDN** (CloudFront/Fastly) in front of public listing pages — even shorter code path.
- For search results, cache the **first page** only. Deep paginations are rare + low-value.

## 4️⃣ Rate-limiting search

An abusive scraper or broken client can DDoS your search tier. Protect with a **token bucket** per user or per IP.

In [7]:
import time

class TokenBucket:
    def __init__(self, rate_per_sec: float, burst: int):
        self.rate = rate_per_sec
        self.capacity = burst
        self.tokens = burst
        self.last = time.monotonic()

    def allow(self) -> bool:
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True
        return False

tb = TokenBucket(rate_per_sec=5, burst=5)   # 5/sec, burst of 5
allowed = sum(tb.allow() for _ in range(20))
print(f"burst: allowed {allowed}/20 in a tight loop (expect ~5)")

time.sleep(1.0)          # let 5 tokens refill
allowed = sum(tb.allow() for _ in range(10))
print(f"after 1s: allowed {allowed}/10 (expect ~5)")

burst: allowed 5/20 in a tight loop (expect ~5)


after 1s: allowed 5/10 (expect ~5)


In production this lives in the API Gateway with a distributed counter (Redis `INCR` + expiry, or a dedicated rate-limit sidecar).

## 🧵 Putting it all together

- **Booking**: DB unique constraint → impossible to double-book.
- **Search**: grid/S2 index + ES → fast geo & date filters.
- **Reads**: cache-aside + single-flight → absorb 100× read traffic.
- **Abuse**: token-bucket rate limit at the edge.

## 🌍 Real-world echoes

- Airbnb's engineering blog describes their "**Availability Service**" as a per-day model fronted by Kafka events — same pattern as our `availability` table.
- They use **dynamic pricing** suggestions for hosts — a batch ML job writes suggestions into the listing DB; CDC flows them into the search index, so surge-priced listings appear at the right rank immediately.
- Search uses **personalisation** (past bookings, price sensitivity) on top of the geo + date filters — another reason to have a dedicated search service rather than SQL.
- **Cancellations** are their own little distributed-system headache: refunds, host penalties, and freeing availability must all succeed. Real systems use an outbox + saga; our `DELETE /bookings/{id}` hides a lot of complexity.

## 🧪 Ideas to explore

1. Add a `price_per_night` that varies by date. Which service owns the pricing rule?
2. Add a **search-as-you-type** endpoint for cities. (See the typeahead lab.)
3. Simulate 10k bookings/sec against the SQLite demo — where does it break, and why?
4. Extend the availability table with a `hold_until` column for a **15-minute cart reservation** (like Airbnb's "Reserve" button before you pay).